## NLP Task 2: Sentiment Analysis 

#### Import Libraries

In [26]:
import pandas as pd
import numpy as np
import re
import string

#Preprocessing and model building,evaluation
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#models imports
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

#nltk libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#### Initializing Raw Data

In [27]:
#Creating dataframe 
data = pd.read_csv("https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv")

data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


#### Describing data

In [28]:
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])
print('\nSentiment Count\n')
print(data['sentiment'].value_counts())
print("\nSample Text:\n", data['review'][0])

Rows: 50000
Columns: 2

Sentiment Count

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Text:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death star

#### NLP Preprocessing (Feature Enginerring)

In [29]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()  # lowercasing
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z]", " ", text)  # remove punctuation/special chars
    
    words = text.split() #tokenization
    words = [w for w in words if w not in stop_words]  # remove stopwords
    words = [stemmer.stem(w) for w in words]  # stemming
    
    return " ".join(words)

data['cleaned'] = data['review'].apply(preprocess)

In [30]:
#Encoding 
data['sentiment'] = data['sentiment'].map({'positive':1, 'negative':0})

In [31]:
X=data['cleaned']
y=data['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
#Bag of Words- Initializing Countvectorizer
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [33]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

###  Model Building


In [34]:
#Logistic Regression

#BOW
lr_bow = LogisticRegression()
lr_bow.fit(X_train_bow, y_train)
y_pred_lr_bow = lr_bow.predict(X_test_bow)

#tf
lr_tf= LogisticRegression()
lr_tf.fit(X_train_tfidf, y_train)
y_pred_lr_tf = lr_tf.predict(X_test_tfidf)

c:\Users\Dell\miniforge3\envs\mix\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [35]:
#Naive Bayes

#BOW
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
y_pred_nb_bow = nb_bow.predict(X_test_bow)

#tf
nb_tf= MultinomialNB()
nb_tf.fit(X_train_tfidf, y_train)
y_pred_nb_tf = nb_tf.predict(X_test_tfidf)

In [36]:
#Decision Tree

#BOW
dt_bow = DecisionTreeClassifier()
dt_bow.fit(X_train_bow, y_train)
y_pred_dt_bow = dt_bow.predict(X_test_bow)

#tf
dt_tf= DecisionTreeClassifier()
dt_tf.fit(X_train_tfidf, y_train)
y_pred_dt_tf = dt_tf.predict(X_test_tfidf)

In [37]:
def evaluate(y_true, y_pred, model_name):
    print(f"\n--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))

#### model evaluation based on Tf-idf

In [38]:
evaluate(y_test, y_pred_lr_tf, "Logistic Regression")
evaluate(y_test, y_pred_nb_tf, "Naive Bayes")
evaluate(y_test, y_pred_dt_tf, "Decision Tree")


--- Logistic Regression ---
Accuracy: 0.8867
Precision: 0.87615562403698
Recall: 0.902758483826156
F1 Score: 0.889258137034503

--- Naive Bayes ---
Accuracy: 0.8519
Precision: 0.8492343934040048
Recall: 0.8585036713633657
F1 Score: 0.8538438764433041

--- Decision Tree ---
Accuracy: 0.7166
Precision: 0.7185332011892963
Recall: 0.7193887676126216
F1 Score: 0.7189607298690995


#### model evaluation based on BOW

In [39]:
evaluate(y_test, y_pred_lr_bow, "Logistic Regression")
evaluate(y_test, y_pred_nb_bow, "Naive Bayes")
evaluate(y_test, y_pred_dt_bow, "Decision Tree")


--- Logistic Regression ---
Accuracy: 0.8732
Precision: 0.8681898066783831
Recall: 0.8823179202222663
F1 Score: 0.8751968503937008

--- Naive Bayes ---
Accuracy: 0.8492
Precision: 0.8547317661241712
Recall: 0.8442151220480254
F1 Score: 0.8494408945686901

--- Decision Tree ---
Accuracy: 0.7168
Precision: 0.7224349929449708
Recall: 0.7112522325858305
F1 Score: 0.7168


Observations:
1. In terms of performance: 
    - Logistic Regression > Naive Bayes > Decision Tree 

2. In comparison between BOW and TF-IDF: 
    - BOW < TF-IDF


Insights:

1. TF-IDF performed better than Bag of Words because it gives importance to meaningful words.
2. Logistic Regression gave the best performance because it works well with high-dimensional sparse data.
3. Naive Bayes was fast but slightly less accurate due to independence assumption.
4. Decision Tree performed worse due to overfitting on text data.



Conclusion:
Logistic Regression + TF-IDF is the best combination for this task.